# Tag EDA

Explore tags & their relationship with songs.

- How many songs per TAG 
- How many songs per TAG group 
- What are popular TAGs

In [6]:
# automatically reload imported modules before executing code

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from nbutils import setup_path

setup_path()

In [8]:
from pyrekordbox import Rekordbox6Database
from pyrekordbox.db6 import tables

# Get table refs 
song_table = tables.DjmdContent
tag_table = tables.DjmdMyTag
song_tag_table = tables.DjmdSongMyTag

In [13]:
import polars as pl
from utils import get_db_content

db = Rekordbox6Database()

clean_songs_df = (
    get_db_content(db)
    .with_columns((pl.col("MyTagNames").list.len() > 0) .alias("has_tags"))
    .filter(
        (pl.col("BPM") > 0) & 
        (pl.col("ArtistName") != "rekordbox")
    ) # Filter sample packs
)

clean_songs_df.head(5)

ContentID,FolderPath,Title,ArtistID,ArtistName,GenreID,GenreName,BPM,DateCreated,Length,MyTagNames,MyTagIDs,has_tags
str,str,str,str,str,str,str,i32,str,i32,list[str],list[str],bool
"""36085625""","""/Users/quintenrosseel/Music/Pi…","""Surrender""","""1673664596""","""Gerd Janson""","""159652946""","""Techno/House""",12200,"""2021-12-26""",413,[],[],false
"""18988943""","""/Users/quintenrosseel/Music/Pi…","""I Want Your Soul""","""1642798025""","""Armand Van Helden""","""3027895697""","""Classics/House""",12800,"""2021-12-26""",399,[],[],false
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"[""80s"", ""Disco"", … ""Chill""]","[""1155275430"", ""62609778"", … ""3718426834""]",true
"""228400738""","""/Users/quintenrosseel/Music/Pi…","""RENT4""","""3953188055""","""Lakim""","""4282047218""","""Garage""",14500,"""2021-07-17""",167,[],[],false
"""217615930""","""/Users/quintenrosseel/Music/Pi…","""Funky Child (1993)""","""1137483595""","""Lords Of The Underground""","""2864790501""","""Hip-hop""",9630,"""2018-10-11""",227,"[""Beats"", ""Hip-Hop"", … ""TAG YEAR""]","[""2053683127"", ""3917148722"", … ""382831235""]",true


In [17]:
tag_schema = {
    "TagID": pl.Utf8, # read as string for joining later on
    "UUID": pl.Utf8,
    "TagGroup": pl.Utf8,
    "TagName": pl.Utf8,
}

clean_tags_df = (
    pl.read_csv("../data/unique_tags.csv", schema=tag_schema)
)

clean_tags_df

TagID,UUID,TagGroup,TagName
str,str,str,str
"""1429694612""","""3e973c3c-001b-4792-92f9-684d40…","""Genre""","""Techno"""
"""2484825285""","""22a59e2a-b3a6-470e-ad4b-92edda…","""Genre""","""Ambient"""
"""3917148722""","""7226a04c-8d3e-4f5a-b6be-acb6c5…","""Genre""","""Hip-Hop"""
"""323305339""","""e8484566-7872-449c-bbb6-e9002b…","""Genre""","""Motown"""
"""385085509""","""c9ab7b83-30a5-4797-b70c-7f18d7…","""Genre""","""Funk"""
…,…,…,…
"""4107364518""","""6a67b8f3-7d9d-4742-b7a9-b5818f…","""Mood""","""Uptempo"""
"""3210010963""","""6869463b-5e4a-49f8-91f8-873b20…","""Mood""","""Guilty"""
"""1018510990""","""3d357a06-cb6c-428a-b41e-b16696…","""Mood""","""Spacy"""


# EDA
- Explode song per MyTagName to have a song-tag association
- Join this tag with their parent
- See the distributions 

In [18]:
clean_eda_df = (
    clean_songs_df
    .explode(["MyTagNames", "MyTagIDs"]) # one tag per song
    .drop("MyTagNames")
    .rename({
        "MyTagIDs": "MyTagID"
    })
    .join(
        clean_tags_df,
        how="left",
        left_on="MyTagID",
        right_on="TagID"
    )
)

clean_eda_df.head(5)

ContentID,FolderPath,Title,ArtistID,ArtistName,GenreID,GenreName,BPM,DateCreated,Length,MyTagID,has_tags,UUID,TagGroup,TagName
str,str,str,str,str,str,str,i32,str,i32,str,bool,str,str,str
"""36085625""","""/Users/quintenrosseel/Music/Pi…","""Surrender""","""1673664596""","""Gerd Janson""","""159652946""","""Techno/House""",12200,"""2021-12-26""",413,null,false,null,null,null
"""18988943""","""/Users/quintenrosseel/Music/Pi…","""I Want Your Soul""","""1642798025""","""Armand Van Helden""","""3027895697""","""Classics/House""",12800,"""2021-12-26""",399,null,false,null,null,null
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"""1155275430""",true,"""e6444280-6cb5-4db9-a791-167f57…","""Years & Origin""","""80s"""
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"""62609778""",true,"""9c39933a-bd9a-45e9-b710-e6cd2c…","""Genre""","""Disco"""
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"""385085509""",true,"""c9ab7b83-30a5-4797-b70c-7f18d7…","""Genre""","""Funk"""
